# 1. Применение RNN для генерации последовательностей
RNN в архитектуре кодировщик-декодировщик используются для задач, где входная последовательность преобразуется в выходную: машинный перевод, генерация описаний изображений, диалоговые системы, транскрипция графем в фонемы. Кодировщик сжимает входную последовательность в скрытое состояние, а декодировщик генерирует выходные токены, на каждом шаге опираясь на ранее сгенерированные.

Стандартное обучение - максимизация правдоподобия (кросс-энтропия) на парах «вход - эталонный выход». Основная проблема такого подхода - сдвиг распределения: на тренировке декодер получает истинные предыдущие токены из эталонной последовательности, а при реальной генерации - свои собственные предсказанные токены. Ошибки накапливаются, и модель становится нестабильной.

Кроме того, у одной входной последовательности может быть множество правильных выходов (например, разных переводов одной фразы). Максимизация правдоподобия единственного референса из датасета неявно наказывает модель за правильные альтернативы, которых нет в обучающих данных. Модель может выучивать высокую вероятность для «мусорных» вариантов, которые хорошо согласуются с эталонными префиксами, но не являются качественными ответами. Supervised обучение требует почти идеального датасета - что на практике редкость.
# 2. Применение RL к дообучению моделей генерации последовательностей

Обучение с подкреплением применяется для дообучения уже предтренированных генеративных моделей, чтобы оптимизировать не правдоподобие, а непосредственно целевую награду (BLEU, CIDEr, человеческие предпочтения). Политикой служит RNN-декодер, состояние - входные данные и сгенерированный контекст, награда вычисляется по финальному выходу.

Прямой policy gradient (REINFORCE) даёт высокую дисперсию градиента. Для её снижения используется Self-Critical Sequence Training (SCST), где базовая линия - это награда, полученная при жадной генерации. Здесь используется именно жадная генерация, потому что сэмплирование шумное и не соответствует продакшн-режиму, а жадный вариант коррелирует с наградой, уменьшает дисперсию и не вносит смещения. SCST позволяет дообучать модель для прямой максимизации метрик качества.

Дополнительные приёмы снижения дисперсии - использование критика в алгоритме Advantage Actor-Critic. На практике сначала делают supervised pre-training, а затем RL fine-tuning, иначе модель страдает от "холодного старта".

RL также применяется для настройки больших языковых моделей под человеческие предпочтения. Трёхшаговый процесс:
*  supervised fine-tuning на демонстрациях

*  обучение модели награды на парных сравнениях ответов людьми

*  оптимизация политики с помощью Proximal Policy Optimization против этой модели награды. Для стабильности добавляется KL-регуляризация, которая не позволяет новой политике слишком сильно отклоняться от исходной SFT-модели.

Более простой и эффективный метод — Direct Preference Optimization. Он исключает явное обучение модели награды и выводит closed-form оптимальной политики:$$\mathcal{L}_{\text{DPO}} = -\log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right)$$
Это уменьшает сложность и повышает стабильность по сравнению с классическим RLHF